# Notebook with code snippets for reading data for two basins for the HBV-SASK model

### Originally, data comes from this source: 
* Razavi, S., Sheikholeslami, R., Gupta, H. V., & Haghnegahdar, A. (2019). VARS-TOOL: A toolbox for comprehensive, efficient, and robust sensitivity and uncertainty analysis. Environmental modelling & software, 112, 95-107.
* Razavi, S., & Gupta, H. V. (2019). A multi-method Generalized Global Sensitivity Matrix approach to accounting for the dynamical nature of earth and environmental systems models. Environmental modelling & software, 114, 1-11. https://doi.org/10.1016/j.envsoft.2018.12.002
* Gupta, Hoshin V. and Razavi, Saman: "Revisiting the Basis of Sensitivity Analysis for Dynamical Earth System Models", Water Resources Research, 2018
* VARS-Tool https://github.com/vars-tool/vars-tool/tree/master/src/varstool/example_models

### This notebook contains code snippets for simple processing and plotting of the data

In [ ]:
import numpy as np
import sys
import pathlib
import pandas as pd
import time

In [ ]:
from uqef_dynamic.utils import utility
from uqef_dynamic.models.hbv_sask import hbvsask_utility as hbv
from uqef_dynamic.models.hbv_sask import HBVSASKModel as hbvmodel

In [ ]:
# importing modules/libs for plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from plotly.offline import plot

### Defining paths

In [ ]:
# TODO - change these paths accordingly
hbv_model_data_path = (
    pathlib.Path.cwd() / ".." / "data" / "HBV-SASK-data"
).resolve()
print(f"hbv_model_data_path-{hbv_model_data_path}")
inputModelDir = hbv_model_data_path

# Examining all the data on our disposal

# Oldman Basin - Precipitation, Temperature, Streamflow

In [ ]:
basin = "Oldman_Basin"
inputModelDir_basin = inputModelDir / basin
precipitation_temperature_inp = inputModelDir_basin / "Precipitation_Temperature.inp"
streamflow_inp = inputModelDir_basin / "streamflow.inp"
workingDir = inputModelDir_basin / "model_runs" / "whole_data_analysis"

In [ ]:
time_column_name = utility.TIME_COLUMN_NAME
precipitation_column_name = "precipitation"
temperature_column_name = "temperature"
streamflow_column_name = "streamflow"

In [ ]:
precipitation_temperature_df = hbv.read_precipitation_temperature(
    precipitation_temperature_inp, 
    time_column_name=time_column_name,
    precipitation_column_name=precipitation_column_name, 
    temperature_column_name=temperature_column_name
)

In [ ]:
streamflow_df = hbv.read_streamflow(
    streamflow_inp, 
    time_column_name=time_column_name, 
    streamflow_column_name=streamflow_column_name
)

In [ ]:
precipitation_temperature_df

In [ ]:
# from datetime import datetime
# temp = datetime.strptime("2006-10-30", '%Y-%m-%d')
# temp.date
# temp = pd.to_datetime("2006-10-30", format='%Y-%m-%d')
# precipitation_temperature_df.loc[(precipitation_temperature_df.index == pd.Timestamp("2006-10-30"))]
# Just one of many ways how one can filter for a desired date/time-stemp
precipitation_temperature_df.loc[(precipitation_temperature_df.index == pd.to_datetime("2006-10-30", format='%Y-%m-%d'))]


In [ ]:
streamflow_df

## examing different initial condition files...

In [ ]:
# reading starting/initial state data
initial_condition_file = inputModelDir_basin / "initial_condition.inp"
initial_condition_data = hbv.read_initial_conditions(initial_condition_file)
print(f"{type(initial_condition_data)} {initial_condition_data}")

In [ ]:
initial_condition_file = inputModelDir_basin / "state_const_df.pkl"  # other options - "state_df.pkl"
initial_condition_df = pd.read_pickle(initial_condition_file, compression="gzip")
initial_condition_df

In [ ]:
# the whole, saved state df; read from the "initial_condition.inp"
initial_condition_file = inputModelDir_basin / "state_df.pkl"  # other options - "state_const_df.pkl"
initial_condition_df = pd.read_pickle(initial_condition_file, compression="gzip")
initial_condition_df

In [ ]:
# by default, the first timestamp will be filtered out/returned...
initial_condition_file = inputModelDir_basin / "state_df.pkl"
timestamp = None
initial_condition_df = hbv.read_initial_conditions(initial_condition_file, timestamp=timestamp)
initial_condition_df

In [ ]:
# one can as well filter based on the specific timestamp
initial_condition_file = inputModelDir_basin / "state_df.pkl"
timestamp = "1979-01-02"
initial_condition_df = hbv.read_initial_conditions(initial_condition_file, timestamp=timestamp)
initial_condition_df

## or, other way is to create a general HBV model object, used just to read forcing data for a specific basin...

In [ ]:
hbvsaskModelObject_Oldman = hbvmodel.HBVSASKModel(
    configurationObject={
        "simulation_settings":{
            "qoi":["Q_cms", "AET"],
            "qoi_column":["Q_cms", "AET"],
            "read_measured_data": ["True", "False"],
            "qoi_column_measured":["streamflow", "None"]
        }
    },
    workingDir=workingDir,
    inputModelDir=inputModelDir,
    basin=basin,
    run_full_timespan=True, 
    writing_results_to_a_file=False,
    plotting=False,
)

In [ ]:
hbvsaskModelObject_Oldman.time_series_measured_data_df

In [ ]:
hbvsaskModelObject_Oldman.read_measured_streamflow

In [ ]:
fig = hbvsaskModelObject_Oldman.plot_input_data(title="Oldman Basin Whole Dataset")
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data.pdf"
fig.write_image(str(plot_filename), format="pdf")

In [ ]:
display(fig)

In [ ]:
hbvsaskModelObject_Oldman.initial_condition_df

In [ ]:
# measure how much time is needed for running one simulation for a whole time-span
start = time.time()
# results_array = hbvsaskModelObject_Oldman(createNewFolder=False)  # this would also work
results_array = hbvsaskModelObject_Oldman.run(createNewFolder=False)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject_Oldman.full_data_range)} days including spin_up_length of {hbvsaskModelObject_Oldman.spin_up_length} days")

In [ ]:
results_array[0][0]['result_time_series']

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject_Oldman.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject_Oldman.time_column_name,
    simulated_time_column=hbvsaskModelObject_Oldman.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject_Oldman.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject_Oldman.precipitation_column_name)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data_measured_streamflow.pdf"
fig.write_image(str(plot_filename), format="pdf")
plot_filename = workingDir / f"forcing_data_measured_streamflow.html"
plot(fig, filename=str(plot_filename), auto_open=False)
display(fig)

In [ ]:
results_array[0][0]['state_df']

In [ ]:
# default/nominal value for the parameters used to run this model simulation...
#hbvsaskModelObject_Oldman
results_array[0][0]['parameters_dict']

In [ ]:
# let's plot also state data
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject_Oldman,
    result_df = results_array[0][0]['result_time_series'],
    state_df = results_array[0][0]['state_df']
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = workingDir / f"forcing_measured_state_streamflow.pdf"
fig.write_image(str(plot_filename), format="pdf", height=1000, width=1100,)
plot_filename = workingDir / f"forcing_measured_state_streamflow.html"
plot(fig, filename=str(plot_filename), auto_open=False)
fig.show()

## Sample time period (via cofiguration file) and re-run the model

In [ ]:
# One way how to filter for a time of interest
basin = "Oldman_Basin"
configurationObject_time = {
    "time_settings":{
      "start_day": 1,
      "start_month": 10,
      "start_year": 2003,
      "end_day": 1,
      "end_month": 10,
      "end_year": 2007,
      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 365
    },
    "simulation_settings":{
        "qoi":["Q_cms", "AET"],
        "qoi_column":["Q_cms", "AET"],
        "read_measured_data": ["True", "False"],
        "qoi_column_measured":["streamflow", "None"]
    }
}
hbvsaskModelObject_Oldman_sampled = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject_time,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basin=basin,
    writing_results_to_a_file=False,
    run_full_timespan=False, 
    plotting=False
)
fig = hbvsaskModelObject_Oldman_sampled.plot_input_data(
    plot_whole_simulation_period=False, 
    fileName=f"forcing_data_{hbvsaskModelObject_Oldman_sampled.start_date_predictions}_{hbvsaskModelObject_Oldman_sampled.end_date}.html",
    title=f"{basin} {hbvsaskModelObject_Oldman_sampled.start_date_predictions}-{hbvsaskModelObject_Oldman_sampled.end_date}")
# fig.update_layout(title=None)
# fig.update_layout(
#     margin=dict(
#         t=10,  # Top margin
#         b=10,  # Bottom margin
#         l=10,  # Left margin
#         r=10   # Right margin
#     )
# )
# plot_filename = workingDir / f"forcing_data_{hbvsaskModelObject_Oldman_sampled.start_date_predictions}_{hbvsaskModelObject_Oldman_sampled.end_date}.pdf"
# fig.write_image(str(plot_filename), format="pdf")
display(fig)

In [ ]:
print(f"start_date - {hbvsaskModelObject_Oldman_sampled.start_date}")
print(f"end_date - {hbvsaskModelObject_Oldman_sampled.end_date}")
print(f"start_date_predictions - {hbvsaskModelObject_Oldman_sampled.start_date_predictions}")


In [ ]:
# measure how much time is needed for running one simulation for a whole time-span
start = time.time()
results_array = hbvsaskModelObject_Oldman_sampled.run(createNewFolder=False)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; \
full_data_range is {len(hbvsaskModelObject_Oldman_sampled.full_data_range)} \
days including spin_up_length of {hbvsaskModelObject_Oldman_sampled.spin_up_length} days")


In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject_Oldman_sampled.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject_Oldman_sampled.time_column_name,
    simulated_time_column=hbvsaskModelObject_Oldman_sampled.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject_Oldman_sampled.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject_Oldman_sampled.precipitation_column_name)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data_measured_streamflow_{hbvsaskModelObject_Oldman_sampled.start_date_predictions}_{hbvsaskModelObject_Oldman_sampled.end_date}.pdf"
fig.write_image(str(plot_filename), format="pdf")
plot_filename = workingDir / f"forcing_data_measured_streamflow_{hbvsaskModelObject_Oldman_sampled.start_date_predictions}_{hbvsaskModelObject_Oldman_sampled.end_date}.html"
plot(fig, filename=str(plot_filename), auto_open=False)
display(fig)

In [ ]:
# let's plot also state data
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject_Oldman_sampled,
    result_df = results_array[0][0]['result_time_series'],
    state_df = results_array[0][0]['state_df']
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = workingDir / f"forcing_measured_state_streamflow_{hbvsaskModelObject_Oldman_sampled.start_date_predictions}_{hbvsaskModelObject_Oldman_sampled.end_date}.pdf"
fig.write_image(str(plot_filename), format="pdf", height=1000, width=1100,)
plot_filename = workingDir / f"forcing_measured_state_streamflow_{hbvsaskModelObject_Oldman_sampled.start_date_predictions}_{hbvsaskModelObject_Oldman_sampled.end_date}.html"
plot(fig, filename=str(plot_filename), auto_open=False)
fig.show()

## Computing some over time statistics of forcing data - again for the whole time-period...

In [ ]:
time_series_measured_data_df = hbvsaskModelObject_Oldman.time_series_measured_data_df

In [ ]:
time_series_measured_data_df

In [ ]:
time_series_measured_data_df.index.name

In [ ]:
print(type(time_series_measured_data_df.index))
if not isinstance(time_series_measured_data_df.index, pd.DatetimeIndex):
    if isinstance(time_series_measured_data_df.index, pd.PeriodIndex):
        time_series_measured_data_df.index = time_series_measured_data_df.index.to_timestamp()
    else:
        time_series_measured_data_df.index = pd.to_datetime(time_series_measured_data_df.index)
    print(type(time_series_measured_data_df.index))


In [ ]:
time_series_measured_data_df.columns

In [ ]:
time_series_measured_data_df.describe().transpose()

In [ ]:
# agg_dict = {column:np.sum for column in list(time_series_measured_data_df.columns)}
agg_dict = {column: "sum" for column in time_series_measured_data_df.columns}

# Resample to annual frequency using year-end
annual_data = time_series_measured_data_df.resample("YE").agg(
    agg_dict, skipna=True, min_count=360
)
# annual_data = time_series_measured_data_df.resample("YE", convention='end').agg(
#     agg_dict, skipna=True, min_count=360
# )
annual_data

In [ ]:
# this only make sense for a precipitation columns
# agg_dict = {"precipitation":np.sum}
agg_dict = {"precipitation": "sum"}
# annual_data_precipitation = time_series_measured_data_df.resample("YE",convention='end').agg(
#     agg_dict, skipna=True, min_count=360
# )
annual_data_precipitation = time_series_measured_data_df.resample("YE").agg(
    agg_dict, skipna=True, min_count=360
)
annual_data_precipitation

In [ ]:
annual_data_precipitation.mean()

# Banff Basin

In [ ]:
basin = "Banff_Basin"
inputModelDir_basin = inputModelDir / basin
workingDir = inputModelDir_basin / "model_runs" / "whole_data_analysis"

In [ ]:
hbvsaskModelObject_Banff = hbvmodel.HBVSASKModel(
    # configurationObject=dict(),
    configurationObject={
        "simulation_settings":{
            "qoi":"Q_cms",
            "qoi_column":"Q_cms",
            "read_measured_data": "True",
            "qoi_column_measured":"streamflow",
        }
    },
    workingDir=workingDir,
    inputModelDir=inputModelDir,
    basin=basin,
    run_full_timespan=True, 
    writing_results_to_a_file=False,
    plotting=False,
)
hbvsaskModelObject_Banff.time_series_measured_data_df

In [ ]:
hbvsaskModelObject_Banff.read_measured_data
hbvsaskModelObject_Banff.read_measured_streamflow

In [ ]:
fig = hbvsaskModelObject_Banff.plot_input_data(title="Banff Basin Whole Dataset")
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data.pdf"
fig.write_image(str(plot_filename), format="pdf")
display(fig)

In [ ]:
# measure how much time is needed for running one simulation for a whole time-span
start = time.time()
results_array = hbvsaskModelObject_Banff.run(createNewFolder=False)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject_Banff.full_data_range)} days including spin_up_length of {hbvsaskModelObject_Banff.spin_up_length} days")

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject_Banff.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject_Banff.time_column_name,
    simulated_time_column=hbvsaskModelObject_Banff.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject_Banff.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject_Banff.precipitation_column_name)

fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data_measured_streamflow.pdf"
fig.write_image(str(plot_filename), format="pdf")
plot_filename = workingDir / f"forcing_data_measured_streamflow.html"
plot(fig, filename=str(plot_filename), auto_open=False)
display(fig)

In [ ]:
hbvsaskModelObject_Banff.initial_condition_df

In [ ]:
time_series_measured_data_df = hbvsaskModelObject_Banff.time_series_measured_data_df

In [ ]:
time_series_measured_data_df.describe().transpose()

In [ ]:
# this only make sense for a precipitation columns
agg_dict = {"precipitation": "sum"}
annual_data_precipitation = time_series_measured_data_df.resample("YE").agg(
    agg_dict, skipna=True, min_count=360
)
# annual_data_precipitation
print(annual_data_precipitation.mean())

In [ ]:
# let's plot also state data
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject_Banff,
    result_df = results_array[0][0]['result_time_series'],
    state_df = results_array[0][0]['state_df']
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = workingDir / f"forcing_measured_state_streamflow.pdf"
fig.write_image(str(plot_filename), format="pdf", height=1000, width=1100,)
plot_filename = workingDir / f"forcing_measured_state_streamflow.html"
plot(fig, filename=str(plot_filename), auto_open=False)
fig.show()

### Analyzing initial state data...

In [ ]:
# the whole, saved state df; read from the "initial_condition.inp"
initial_condition_file = inputModelDir_basin / "state_df.pkl"  # other options - "state_const_df.pkl"
initial_condition_df = pd.read_pickle(initial_condition_file, compression="gzip")
initial_condition_df

In [ ]:
# by default, the first timestamp will be filtered out/returned...
initial_condition_file = inputModelDir_basin / "state_df.pkl"
timestamp = None
initial_condition_df = hbv.read_initial_conditions(initial_condition_file, timestamp=timestamp)
initial_condition_df

In [ ]:
# one can as well filter based on the specific timestamp
initial_condition_file = inputModelDir_basin / "state_df.pkl"
timestamp = "1979-01-02"
initial_condition_df = hbv.read_initial_conditions(initial_condition_file, timestamp=timestamp)
initial_condition_df

## Look closer to a specific time-span

In [ ]:
# Zooming-in into the period of interest

# to make consecutive runs 1.10.2005-1.10.2007
# time 1.10.2002-(1096)-1.10.2005-(365)-1.10.2006
# time 2.10.2003-(1096)-2.10.2006-(364)-1.10.2007

basin = "Banff_Basin"
configurationObject_time = {
    "time_settings":
    {
      "start_day": 2,
      "start_month": 10,
      "start_year": 2002,
      "end_day": 1,
      "end_month": 10,
      "end_year": 2007,
      "run_full_timespan":"False",
      "spin_up_length":1096,
      "simulation_length": 700,
    },
    # "time_settings":{
    #   "start_day": 1, #1
    #   "start_month": 10, #10
    #   "start_year": 2001, #2002
    #   "end_day": 1,  #1
    #   "end_month": 10,  #10
    #   "end_year": 2007,  #2007
    #   "run_full_timespan":"False",
    #   "spin_up_length":1096,
    #   "simulation_length": 2190 #730
    # },
    "simulation_settings":{
        "qoi":["Q_cms", "AET"],
        "qoi_column":["Q_cms", "AET"],
        "read_measured_data": ["True", "False"],
        "qoi_column_measured":["streamflow", "None"]
    }
}
hbvsaskModelObject_Banff_sampled = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject_time,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basin=basin,
    writing_results_to_a_file=False,
    run_full_timespan=False, 
    plotting=False
)

print(f"start_date: {hbvsaskModelObject_Banff_sampled.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject_Banff_sampled.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject_Banff_sampled.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject_Banff_sampled.full_data_range)} days including spin_up_length of {hbvsaskModelObject_Banff_sampled.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject_Banff_sampled.simulation_range)} days")

fig = hbvsaskModelObject_Banff_sampled.plot_input_data(
    plot_whole_simulation_period=False, 
    fileName=f"forcing_data_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.html",
    title=f"{basin} {hbvsaskModelObject_Banff_sampled.start_date_predictions}-{hbvsaskModelObject_Banff_sampled.end_date}")

fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
# plot_filename = workingDir / f"forcing_data_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.pdf"
# fig.write_image(str(plot_filename), format="pdf")
display(fig)

In [ ]:
# Zooming-in into the period of interest

# to make consecutive runs 1.10.2005-1.10.2007
# time 1.10.2001-(1096)-1.10.2004-(365)-1.10.2005
# time 2.10.2002-(1096)-2.10.2005-(364)-1.10.2006
# time 2.10.2003-(1096)-2.10.2006-(364)-1.10.2007
# time 1.10.2004-(1096)-2.10.2007-(365)-1.10.2008
# time 2.10.2005-(1096)-2.10.2008-(364)-1.10.2009
# time 2.10.2006-(1096)-2.10.2009-(364)-1.10.2010

basin = "Banff_Basin"
configurationObject_time = {
    "time_settings":{
      "start_day": 2, #1
      "start_month": 10, #10
      "start_year": 2006, #2002
      "end_day": 1,  #1
      "end_month": 10,  #10
      "end_year": 2010,  #2007
      "run_full_timespan":"False",
      "spin_up_length":1096,
      "simulation_length": 364#2190 #730
    },
    "simulation_settings":{
        "qoi":["Q_cms", "AET"],
        "qoi_column":["Q_cms", "AET"],
        "read_measured_data": ["True", "False"],
        "qoi_column_measured":["streamflow", "None"]
    }
}
hbvsaskModelObject_Banff_sampled = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject_time,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basin=basin,
    writing_results_to_a_file=False,
    run_full_timespan=False, 
    plotting=False
)

print(f"start_date - {hbvsaskModelObject_Banff_sampled.start_date}")
print(f"end_date - {hbvsaskModelObject_Banff_sampled.end_date}")
print(f"start_date_predictions - {hbvsaskModelObject_Banff_sampled.start_date_predictions}")


fig = hbvsaskModelObject_Banff_sampled.plot_input_data(
    plot_whole_simulation_period=False, 
    fileName=f"forcing_data_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.html",
    title=f"{basin} {hbvsaskModelObject_Banff_sampled.start_date_predictions}-{hbvsaskModelObject_Banff_sampled.end_date}")

fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.pdf"
fig.write_image(str(plot_filename), format="pdf")
display(fig)

In [ ]:
# measure how much time is needed for running one simulation for a whole time-span
start = time.time()
results_array = hbvsaskModelObject_Banff_sampled.run(createNewFolder=False)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; \
full_data_range is {len(hbvsaskModelObject_Banff_sampled.full_data_range)} \
days including spin_up_length of {hbvsaskModelObject_Banff_sampled.spin_up_length} days")


In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject_Banff_sampled.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject_Banff_sampled.time_column_name,
    simulated_time_column=hbvsaskModelObject_Banff_sampled.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject_Banff_sampled.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject_Banff_sampled.precipitation_column_name)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data_measured_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.pdf"
fig.write_image(str(plot_filename), format="pdf")
plot_filename = workingDir / f"forcing_data_measured_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.html"
plot(fig, filename=str(plot_filename), auto_open=False)
display(fig)

In [ ]:
# let's plot also state data
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject_Banff_sampled,
    result_df = results_array[0][0]['result_time_series'],
    state_df = results_array[0][0]['state_df']
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = workingDir / f"forcing_measured_state_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.pdf"
fig.write_image(str(plot_filename), format="pdf", height=1000, width=1100,)
plot_filename = workingDir / f"forcing_measured_state_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.html"
plot(fig, filename=str(plot_filename), auto_open=False)
fig.show()